# Tweety-3 — Argumentation abstraite de Dung en C#/.NET (port natif IKVM)

> **Série Tweety — port C#/.NET natif (EPIC [#4667](https://github.com/jsboige/CoursIA/issues/4667)).**
> Ce notebook exploite le module **`arg-dung`** de [TweetyProject](https://tweetyproject.org/) **sans JVM** :
> la librairie Java est compilée vers un fat-jar Maven shade puis exécutée sur le runtime .NET via
> [IKVM](https://github.com/ikvm-refined/ikvm).

Navigation : [Tweety-2-Basic-Logics-Csharp](Tweety-2-Basic-Logics-Csharp.ipynb) (propositionnel) ·
[Tweety-2b-Semantics-Csharp](Tweety-2b-Semantics-Csharp.ipynb) (mondes possibles) ·
[Tweety-2c-FOL-Csharp](Tweety-2c-FOL-Csharp.ipynb) (premier ordre) ·
**Tweety-3-Dung (ce notebook)**.

---

## Objectifs pédagogiques

L'**argumentation abstraite de Dung** (1995) est le formalisme historique fondateur de TweetyProject.
On s'affranchit de la nature des arguments pour ne retenir que leur **structure d'attaque** :

- un **argument** est un nœud abstrait (on ne regarde pas son contenu) ;
- une **attaque** `a → b` (« `a` réfute `b` ») est une arête orientée ;
- un **cadre d'argumentation** (AF, *argumentation framework*) = un graphe orienté d'attaques.

La question centrale : étant donné un AF, **quels arguments accepter** ? Dung définit plusieurs
**sémantiques** (ensembles d'arguments « rationnellement acceptables » = les *extensions*) :

| Sémantique | Idée | Extensions |
|------------|------|------------|
| **Sans conflit** (CF) | pas d'attaque mutuelle interne | plusieurs |
| **Admissible** | sans conflit + se défend contre toute attaque | plusieurs |
| **Complète** (CO) | admissible + contient tout argument qu'elle défend | plusieurs |
| **Fondée** (GR, *grounded*) | la plus petite extension complète | **une seule** |
| **Préférée** (PR) | une extension complète **maximale** (pour l'inclusion) | ≥ 1 |
| **Stable** (ST) | bat (attaque) tout argument hors de l'extension | 0 ou plusieurs |

Ce notebook construit des AF en C#, calcule leurs extensions sous chaque sémantique, et teste
l'**acceptation** (sceptique) d'un argument.



## 1 — Runtime IKVM : charger le module `arg-dung`

On installe le runtime IKVM, on fusionne l'image (base + arch), puis on charge la DLL
`org.tweetyproject.tweety-dung.dll` (compilée côté build à partir d'un fat-jar shade embarquant
`arg-dung` + ses dépendances transitives : `graphs`, `commons`, `math`, `logics-fol`, `logics-pl`).



In [1]:
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "nuget: IKVM.Image.runtime.win-x64, 8.15.0"



The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages IKVM, 8.15.0 IKVM.Image, 8.15.0

In [2]:
using System.IO;
string ikvmVer = "8.15.0", ikvmRid = "win-x64";
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);
void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}
if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
AppContext.SetData("IKVM.Home", ikvmHome);
Console.WriteLine("IKVM home=" + (File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat")) ? "OK" : "MISSING"));



IKVM home=OK


In [3]:
#r "org.tweetyproject.tweety-dung.dll"



## 2 — Construire un cadre d'argumentation (AF)

Un AF se construit avec `DungTheory` (qui implémente `Graph<Argument>`). On y ajoute des
**arguments** (`Argument`, nommés) et des **attaques** (`Attack(attacker, attacked)`).

### 2.1 Un premier AF : attaque simple `a → b`

L'argument `a` attaque `b`. Intuitivement, si on accepte `a`, on ne peut pas accepter `b`.



In [4]:
using org.tweetyproject.arg.dung.syntax;
using org.tweetyproject.arg.dung.semantics;
using org.tweetyproject.arg.dung.reasoner;
using System.Collections.Generic;

var a = new Argument("a");
var b = new Argument("b");

var af = new DungTheory();
af.add(a);
af.add(b);
af.add(new Attack(a, b));   // a attaque b

Console.WriteLine("AF = " + af);
Console.WriteLine("nb noeuds = " + af.getNodes().size());



AF = <{ a, b },[(a,b)]>


nb noeuds = 2


### 2.2 Attaque mutuelle et cycle

Quand `a` et `b` s'attaquent mutuellement, aucune des deux sémantiques « strictes » ne peut trancher
seule : c'est le cas typique où **plusieurs extensions** coexistent (l'une accepte `a`, l'autre `b`).
On construit aussi un cycle `c → d → e → c`.



In [5]:
var af2 = new DungTheory();
var x = new Argument("x");
var y = new Argument("y");
af2.add(x); af2.add(y);
af2.add(new Attack(x, y));
af2.add(new Attack(y, x));   // attaque mutuelle
Console.WriteLine("AF2 (attaque mutuelle) = " + af2);
Console.WriteLine("Bien-fondé ? " + af2.isWellFounded());   // faux : cycle d'attaque



AF2 (attaque mutuelle) = <{ x, y },[(x,y), (y,x)]>


Bien-fondé ? False


## 3 — Calculer les extensions

Chaque sémantique est incarnée par un **raisonneur** (`Simple*Reasoner`). Tous héritent de
`AbstractExtensionReasoner` qui expose :

- **`getModels(DungTheory)`** → la collection des **extensions** (`Extension<DungTheory>`) ;
- **`query(DungTheory, Argument)`** → `Boolean`, acceptation **sceptique** de l'argument.

### 3.1 L'extension fondée (grounded) — unique, sceptique

L'extension **fondée** est la plus prudente : on part de l'ensemble vide, on ajoute tout argument
non attaqué, puis tout argument défendu, et on itère jusqu'à point fixe. Elle est **toujours unique**.



In [6]:
// Sur l'AF a -> b : l'extension fondée est {a} (a est inattaqué, b est battu par a)
AbstractExtensionReasoner grounded = new SimpleGroundedReasoner();
// getModels renvoie une java.util.Collection : on itère via toArray() (foreach C# cast chaque élément).
Console.WriteLine("Extensions fondées de AF :");
foreach (Extension ext in grounded.getModels(af).toArray(new Extension[0]))
    Console.WriteLine("  " + ext);

// Acceptation sceptique : a accepté, b refusé
Console.WriteLine("\nAF, fondée ⊨ a ? " + grounded.query(af, a));
Console.WriteLine("AF, fondée ⊨ b ? " + grounded.query(af, b));



Extensions fondées de AF :


  {a}



AF, fondée ⊨ a ? true


AF, fondée ⊨ b ? false


### 3.2 Attaque mutuelle : fondée vs stable vs préférée

Sur l'AF à attaque mutuelle `{x↔y}` :

- **fondée** : `{}` (vide — ni x ni y n'est défendu, le point fixe ne décolle pas) ;
- **stable** : `{x}` et `{y}` (deux extensions — chaque argument bat l'autre) ;
- **préférée** : `{x}` et `{y}` (deux extensions maximales admissibles).

C'est l'exemple canonique où sémantiques **divergent** : prudente (fondée) vs tranchantes (stable/préférée).



In [7]:
void Dump(string label, AbstractExtensionReasoner r, DungTheory theory)
{
    Console.WriteLine(label + " :");
    foreach (Extension ext in r.getModels(theory).toArray(new Extension[0]))
        Console.WriteLine("  " + ext);
}

Dump("AF2 fondée",   new SimpleGroundedReasoner(),  af2);
Dump("AF2 stable",   new SimpleStableReasoner(),    af2);
Dump("AF2 préférée", new SimplePreferredReasoner(), af2);
Dump("AF2 complète", new SimpleCompleteReasoner(),  af2);



AF2 fondée :


  {}


AF2 stable :


  {x}


  {y}


...

  {y}


AF2 complète :


  {x}


  {y}


  {}


## 4 — Un AF plus riche : défense en chaîne

Construisons un AF où un argument est attaqué mais **défendu** par un autre :

```
        d
        |
        v
   c <- a -> b
```

- `a` attaque `b` et `c` ;
- `d` attaque `a` (donc `d` défend indirectement `b` et `c`).

L'extension fondée devrait contenir `d` (inattaqué), qui défend `b` et `c` contre `a`.



In [8]:
var A = new Argument("A");
var B = new Argument("B");
var C = new Argument("C");
var D = new Argument("D");

var af3 = new DungTheory();
af3.add(A); af3.add(B); af3.add(C); af3.add(D);
af3.add(new Attack(A, B));   // A attaque B
af3.add(new Attack(A, C));   // A attaque C
af3.add(new Attack(D, A));   // D attaque A (défend B et C)
Console.WriteLine("AF3 = " + af3);

// Extension fondée : D (inattaqué) défend B et C contre A -> {D, B, C}
AbstractExtensionReasoner gr = new SimpleGroundedReasoner();
foreach (Extension ext in gr.getModels(af3).toArray(new Extension[0]))
    Console.WriteLine("AF3 fondée = " + ext);

// Acceptation sceptique
Console.WriteLine("\nAF3 fondée ⊨ D ? " + gr.query(af3, D));
Console.WriteLine("AF3 fondée ⊨ B ? " + gr.query(af3, B));
Console.WriteLine("AF3 fondée ⊨ A ? " + gr.query(af3, A));   // battu par D



AF3 = <{ A, B, C, D },[(A,B), (A,C), (D,A)]>


AF3 fondée = {B,C,D}



AF3 fondée ⊨ D ? true


AF3 fondée ⊨ B ? true


AF3 fondée ⊨ A ? false


---

## Exercices

> Stubs **sans `throw`/`raise`** (convention C.1) : le notebook s'exécute de bout en bout même non complété.



### Exercice 1 — Cycle à trois : stable vs fondée

Construisez un AF en cycle `p → q → r → p` (chaque argument en attaque un autre, formant un 3-cycle).
Affichez l'extension **fondée** et les extensions **stables**.

**Question** : combien d'extensions stables obtient-on ? L'extension fondée est-elle vide ? (Indice :
dans un cycle impair sans argument inattaqué, le point fixe fondé ne décolle pas, mais chaque argument
isolé bat la chaîne restante → plusieurs stables.)



In [9]:
// TODO etudiant : cycle p -> q -> r -> p
// Indice : declarez p,q,r, ajoutez 3 Attack, puis getModels(SimpleGroundedReasoner) et (SimpleStableReasoner).
object nbStables = null;  // TODO etudiant : nombre d'extensions stables (int)
Console.WriteLine("extensions stables du cycle = " + (nbStables ?? "Exercice a completer"));



extensions stables du cycle = Exercice a completer


### Exercice 2 — Auto-attaque

Un argument qui **s'attaque lui-même** (`s → s`) est dit *auto-réfuté*. Construisez un AF avec un seul
argument `s` qui s'attaque, et vérifiez que **aucune** sémantique admissible ne l'accepte :
l'extension fondée doit être vide, et `query(af, s)` doit valoir `false`.

**Indice** : `af.add(new Attack(s, s))`.



In [10]:
// TODO etudiant : auto-attaque s -> s
// Indice : un seul Argument s + Attack(s,s). Puis grounded.query(af, s).
bool? accepteS = null;  // TODO etudiant
Console.WriteLine("AF auto-attaque, fondée ⊨ s ? " + (accepteS?.ToString() ?? "Exercice a completer"));



AF auto-attaque, fondée ⊨ s ? Exercice a completer


### Exercice 3 — Comparer complète et préférée

Sur l'AF `{a → b, a → c}` (où `a` attaque `b` et `c`, mais rien n'attaque `a`), la sémantique
**complète** admet une extension non-maximale en plus de la préférée. Affichez les extensions
**complètes** ET **préférées** et identifiez laquelle est commune.

**Indice** : `{a}` est complète ; est-elle aussi préférée ? Et `{a, b}` est-elle admissible
(si `b` et `c` ne s'attaquent pas mutuellement) ?



In [11]:
// TODO etudiant : AF {a -> b, a -> c}, extensions completes vs preferees
// Indice : utilisez Dump("complete", new SimpleCompleteReasoner(), af) et ("preferee", new SimplePreferredReasoner(), af).
string verdict = null;  // TODO etudiant : quelle extension est commune aux deux sémantiques ?
Console.WriteLine("extension commune complète/préférée = " + (verdict ?? "Exercice a completer"));



extension commune complète/préférée = Exercice a completer


---

## Conclusion

On a porté en C#/.NET natif (sans JVM) l'**argumentation abstraite de Dung** — le cœur historique de
TweetyProject — via le module `arg-dung` et IKVM :

1. **Construction** d'un cadre d'argumentation (`DungTheory` + `Argument` + `Attack`), y compris
   attaques mutuelles, cycles et défense indirecte.
2. **Calcul des extensions** sous les quatre sémantiques fondatrices : fondée (unique, sceptique),
   complète, préférée (maximale), stable (agressive) — via `getModels`.
3. **Acceptation sceptique** d'un argument via `query`.

### Pourquoi plusieurs sémantiques ?

| Sémantique | Stratégie | Quand l'utiliser |
|------------|-----------|------------------|
| **Fondée** | la plus prudente (point fixe) | décision unique, conservatrice |
| **Complète** | tout argument défendu est inclus | fermeture « naturelle » |
| **Préférée** | maximale pour l'inclusion | « le mieux possible » sans arbitrer |
| **Stable** | bat tout l'extérieur | verdicts nets, mais peut ne pas exister |

### Références

- Dung, P. M. *On the Acceptability of Arguments and its Fundamental Role in Nonmonotonic Reasoning,
  Logic Programming and n-Person Games*. Artificial Intelligence, 1995.
- TweetyProject — [arg-dung module](https://tweetyproject.org/api/dung/).
- Port C#/.NET via IKVM — EPIC [#4667](https://github.com/jsboige/CoursIA/issues/4667).

